# Make predictions with Onunet

In [1]:
from tensorflow.keras.utils import set_random_seed
from skimage import img_as_ubyte
import os
import re

import pandas as pd
import sys
from pathlib import Path
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.ticker import MultipleLocator

sys.path.append('/home/jovyan/projects/onunet')
from model import *
from data import *

2025-10-23 18:57:16.553606: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-23 18:57:16.757828: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-23 18:57:16.759343: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-23 18:57:18.142712: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Set the seed using keras.utils.set_random_seed. This will set:
# 1) `numpy` seed
# 2) backend random seed
# 3) `python` random seed
set_random_seed(812)

Define paths:

In [3]:
print(os.getcwd())
os.chdir('/home/jovyan/projects/onunet')
print(os.getcwd())
species = "Du25"
path2data = "/home/jovyan/kDrive/cellcount/data"
path2tiles = os.path.join("/home/jovyan/kDrive/cellcount/data/preprocessing", species, "ON1_1/tiles_512x512")
#path2tiles = os.path.join(path2data, "preprocessing", species, "ON1_1/sample")
print(path2tiles)
path2prediction = os.path.join(path2data, "onunet/optical_nerve/prediction", species, "tiles")
print(path2prediction)
path2checkpoints = "/home/jovyan/kDrive/cellcount/data/onunet/model_weights/06_09_2024"
print(path2checkpoints)

/home/jovyan
/home/jovyan/projects/onunet
/home/jovyan/kDrive/cellcount/data/preprocessing/Du25/ON1_1/tiles_512x512
/home/jovyan/kDrive/cellcount/data/onunet/optical_nerve/prediction/Du25/tiles
/home/jovyan/kDrive/cellcount/data/onunet/model_weights/06_09_2024


In [4]:
def predictGenerator(
    predict_path,
    num_image,
    target_size = (256, 256),
    flag_multi_class = False,
    as_gray = True):

    basepath = Path(predict_path)
    tiles = (entry for entry in basepath.iterdir() if entry.is_file() and entry.name.startswith('tile') and entry.name.endswith('.png')) # generator
    tiles_names = sorted([entry.name for entry in tiles])
    for tile in tiles_names:
        img = io.imread(os.path.join(predict_path, tile), as_gray = as_gray)
        img = img / 255.
        img = trans.resize(img, target_size)
        img = np.reshape(img, img.shape + (1,)) if (not flag_multi_class) else img
        img = np.reshape(img, (1,) + img.shape)
        yield img

def tile_coordinates(path2directory):
    basepath = Path(path2directory)
    tiles = (entry for entry in basepath.iterdir() if entry.is_file() and entry.name.startswith('tile') and entry.name.endswith('.png')) # generator
    tiles_names = sorted([entry.name for entry in tiles])
    tile_coords = []
    for tile in tiles_names:    
        tile_name_splitted = re.split('_|\.', tile)
        i = tile_name_splitted[1]
        j = tile_name_splitted[2]
        tile_coords.append((int(i),int(j)))
    
    return tile_coords

def saveResult(save_path, npyfile, tile_coordinates=None):
    for i, item in enumerate(npyfile):
        img = item[:,:,0]
        if tile_coordinates is not None:
            filename = f"predict_{tile_coordinates[i][0]}_{tile_coordinates[i][1]}.png"
        else: # string formatting for membrane dataset
            filename = f"{i}_predict.png"
        io.imsave(os.path.join(save_path, filename), img_as_ubyte(img))

def countTiles(path2directory):
    basepath = Path(path2directory)
    tiles = (entry for entry in basepath.iterdir() if entry.is_file() and entry.name.startswith('tile') and entry.name.endswith('.png')) # generator
    tiles_names = [entry.name for entry in tiles]
    return len(tiles_names)


In [5]:
num_image = countTiles(path2tiles)
predict_generator = predictGenerator(path2tiles, num_image=num_image, target_size = (512, 512))
tile_coords = tile_coordinates(path2tiles)
#model = unet()
model = weighted_loss_unet(input_shape=(512, 512, 1), inference=True)
print(path2checkpoints)
model.load_weights(os.path.join(path2checkpoints, "optical_nerve_weights.hdf5"))
results = model.predict(predict_generator, verbose=1)
saveResult(path2prediction, results, tile_coords)
print(tile_coords)

/home/jovyan/kDrive/cellcount/data/onunet/model_weights/06_09_2024
241/241 [==============================] - 511s 2s/step
[(10, 0), (10, 1), (10, 10), (10, 11), (10, 12), (10, 13), (10, 14), (10, 15), (10, 16), (10, 17), (10, 18), (10, 2), (10, 3), (10, 4), (10, 5), (10, 6), (10, 7), (10, 8), (10, 9), (11, 0), (11, 1), (11, 10), (11, 11), (11, 12), (11, 13), (11, 14), (11, 15), (11, 16), (11, 17), (11, 2), (11, 3), (11, 4), (11, 5), (11, 6), (11, 7), (11, 8), (11, 9), (12, 1), (12, 10), (12, 11), (12, 12), (12, 13), (12, 14), (12, 15), (12, 16), (12, 17), (12, 2), (12, 3), (12, 4), (12, 5), (12, 6), (12, 7), (12, 8), (12, 9), (13, 1), (13, 10), (13, 11), (13, 12), (13, 13), (13, 14), (13, 15), (13, 16), (13, 2), (13, 3), (13, 4), (13, 5), (13, 6), (13, 7), (13, 8), (13, 9), (14, 1), (14, 10), (14, 11), (14, 12), (14, 13), (14, 14), (14, 2), (14, 3), (14, 4), (14, 5), (14, 6), (14, 7), (14, 8), (14, 9), (15, 10), (15, 11), (15, 12), (15, 2), (15, 3), (15, 4), (15, 5), (15, 6), (15, 7),